# 01A — Petrobras 3W Pack

**Outcome:** translate a hash-pinned, real-well Petrobras 3W fixture into the
same standardized pack interface used by telecom.

This notebook owns 3W-specific facts: the official source inventory, native
Portuguese field names, units declared in `dataset.ini`, real-well file
selection, and the meanings of `class` and `state`. It does not invent
manifolds, maintenance tickets, severity, or cross-well causality.


## 1. Setup

The official 3W 2.0.0 folder is expected at:

`MyDrive/anomaly_detection/sources/petrobras_3w/2.0.0/raw/3w_dataset_2.0.0/`

The pack deliberately uses three **real WELL** files, not simulated or
hand-drawn files. Their purpose is to challenge the contract, not to train the
final model.


In [ ]:
import configparser
import os
import shutil
import sys
import tempfile
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_ROOT = Path(os.getenv(
    "ANOMALY_DRIVE_ROOT",
    "/content/drive/MyDrive/anomaly_detection",
))
NOTEBOOK_HOME = Path(os.getenv(
    "ANOMALY_NOTEBOOK_HOME",
    DRIVE_ROOT / "research" / "week1",
))
if str(NOTEBOOK_HOME) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_HOME))

from week1_core import (
    EVAL_SCHEMAS,
    PACK_METRIC_SCHEMA,
    finalise_pack,
    immutable_directory,
    pack_core_hashes,
    read_json,
    source_record,
)

SOURCE = Path(os.getenv(
    "THREEW_SOURCE_ROOT",
    DRIVE_ROOT / "sources" / "petrobras_3w" / "2.0.0"
    / "raw" / "3w_dataset_2.0.0",
))
PACK_RUN_ID = os.getenv(
    "THREEW_PACK_RUN_ID",
    "real_well_contract_fixture_v1",
)
PACK_ROOT = (
    DRIVE_ROOT / "outputs" / "packs" / "petrobras_3w" / PACK_RUN_ID
)
BATCH_ROWS = int(os.getenv("THREEW_BATCH_ROWS", "5000"))
RUN_BUILD = os.getenv("RUN_THREEW_PACK", "1") == "1"

display(pd.Series({
    "source": str(SOURCE),
    "pack_root": str(PACK_ROOT),
    "batch_rows": BATCH_ROWS,
    "source_kind": "real wells only",
}, name="value").to_frame())


## 2. The Petrobras 3W phrasebook

The standardized names below are readable English identifiers. Units are
taken directly from the official `dataset.ini`: pressure in Pa, flow in m³/s,
temperature in °C, openings in percent, and valve state as a source code.

`class` and `state` are intentionally absent because they are evaluation
labels, not model inputs.


In [ ]:
METRIC_COLUMNS = ["native_field", *PACK_METRIC_SCHEMA]
metric_map = pd.DataFrame([
    ("ABER-CKGL", "gas_lift_choke_opening", "oil_well", "gauge", "percent", None),
    ("ABER-CKP", "production_choke_opening", "oil_well", "gauge", "percent", None),
    ("ESTADO-DHSV", "downhole_safety_valve_state", "oil_well", "discrete_state", "state_code", None),
    ("ESTADO-M1", "production_master_valve_state", "oil_well", "discrete_state", "state_code", None),
    ("ESTADO-M2", "annulus_master_valve_state", "oil_well", "discrete_state", "state_code", None),
    ("ESTADO-PXO", "pig_crossover_valve_state", "oil_well", "discrete_state", "state_code", None),
    ("ESTADO-SDV-GL", "gas_lift_shutdown_valve_state", "oil_well", "discrete_state", "state_code", None),
    ("ESTADO-SDV-P", "production_shutdown_valve_state", "oil_well", "discrete_state", "state_code", None),
    ("ESTADO-W1", "production_wing_valve_state", "oil_well", "discrete_state", "state_code", None),
    ("ESTADO-W2", "annulus_wing_valve_state", "oil_well", "discrete_state", "state_code", None),
    ("ESTADO-XO", "crossover_valve_state", "oil_well", "discrete_state", "state_code", None),
    ("P-ANULAR", "annulus_pressure", "oil_well", "gauge", "Pa", None),
    ("P-JUS-BS", "service_pump_downstream_pressure", "oil_well", "gauge", "Pa", None),
    ("P-JUS-CKGL", "gas_lift_choke_downstream_pressure", "oil_well", "gauge", "Pa", None),
    ("P-JUS-CKP", "production_choke_downstream_pressure", "oil_well", "gauge", "Pa", None),
    ("P-MON-CKGL", "gas_lift_choke_upstream_pressure", "oil_well", "gauge", "Pa", None),
    ("P-MON-CKP", "production_choke_upstream_pressure", "oil_well", "gauge", "Pa", None),
    ("P-MON-SDV-P", "production_shutdown_valve_upstream_pressure", "oil_well", "gauge", "Pa", None),
    ("P-PDG", "downhole_pressure", "oil_well", "gauge", "Pa", None),
    ("PT-P", "production_tube_downstream_pressure", "oil_well", "gauge", "Pa", None),
    ("P-TPT", "tubing_pressure", "oil_well", "gauge", "Pa", None),
    ("QBS", "service_pump_flow_rate", "oil_well", "gauge", "m3/s", None),
    ("QGL", "gas_lift_flow_rate", "oil_well", "gauge", "m3/s", None),
    ("T-JUS-CKP", "production_choke_downstream_temperature", "oil_well", "gauge", "degC", None),
    ("T-MON-CKP", "production_choke_upstream_temperature", "oil_well", "gauge", "degC", None),
    ("T-PDG", "downhole_temperature", "oil_well", "gauge", "degC", None),
    ("T-TPT", "tubing_temperature", "oil_well", "gauge", "degC", None),
], columns=METRIC_COLUMNS)

assert len(metric_map) == 27
assert metric_map["metric_id"].is_unique
assert {"class", "state"}.isdisjoint(metric_map["native_field"])
display(metric_map)


## 3. Verify the official source and the pinned real-well fixture

The official release has version `2.0.0`, event directories `0`–`9`, and
2,228 event-instance Parquet files. The three files below were previously
verified as the smallest deterministic real-well subset covering:

- normal operation;
- a persistent event;
- a transient event with a state transition;
- missing measurements.

The assertions prevent a duplicate or incomplete upload from being used.


In [ ]:
EXPECTED_FILES = 2_228
SELECTED_FILES = [
    "0/WELL-00001_20170201160311.parquet",
    "3/WELL-00014_20170917140000.parquet",
    "8/WELL-00019_20210617032654.parquet",
]

parser = configparser.ConfigParser()
parser.read(SOURCE / "dataset.ini", encoding="utf-8")
version = parser.get("VERSION", "DATASET")
event_files = [
    path
    for event_code in range(10)
    for path in (SOURCE / str(event_code)).glob("*.parquet")
]
missing_directories = [
    str(code) for code in range(10)
    if not (SOURCE / str(code)).is_dir()
]
inventory = {
    "version": version,
    "event_file_count": len(event_files),
    "missing_event_directories": missing_directories,
    "selected_files_exist": all(
        (SOURCE / relative).is_file()
        for relative in SELECTED_FILES
    ),
}
display(pd.Series(inventory, name="value").to_frame())

assert version == "2.0.0"
assert len(event_files) == EXPECTED_FILES
assert not missing_directories
assert inventory["selected_files_exist"]
assert all(Path(name).name.startswith("WELL-") for name in SELECTED_FILES)

selection = []
for relative in SELECTED_FILES:
    path = SOURCE / relative
    parquet = pq.ParquetFile(path)
    frame = pd.read_parquet(path, columns=["class", "state"])
    selection.append({
        "file": relative,
        "entity_id": path.name.split("_", 1)[0],
        "event_code": int(path.parent.name),
        "rows": parquet.metadata.num_rows,
        "has_null_measurement": any(
            (parquet.metadata.row_group(group).column(column).statistics
             is not None)
            and (
                parquet.metadata.row_group(group).column(column)
                .statistics.null_count or 0
            ) > 0
            for group in range(parquet.metadata.num_row_groups)
            for column in range(27)
        ),
        "state_codes": sorted(
            str(value) for value in frame["state"].dropna().unique()
        ),
    })

selection_table = pd.DataFrame(selection)
display(selection_table)
assert set(selection_table["event_code"]) == {0, 3, 8}
assert selection_table["has_null_measurement"].any()
assert len(set().union(
    *(set(values) for values in selection_table["state_codes"])
)) > 1


## 4. Build the standardized Petrobras 3W Pack

Measurements are written to `PACK-CORE` as wide, one-second observations.
The source `class` field becomes event truth; `state` becomes condition-state
truth. Both are written only to `PACK-EVAL`.

The relation table is empty because 3W does not provide physical manifold or
cross-well topology. Empty is more honest than invented structure.


In [ ]:
def event_descriptions(source):
    parser = configparser.ConfigParser()
    parser.read(source / "dataset.ini", encoding="utf-8")
    names = [
        item.strip()
        for item in parser.get("EVENTS", "NAMES")
        .replace("\n", "").split(",")
    ]
    return {
        parser.getint(name, "LABEL"): parser.get(name, "DESCRIPTION")
        for name in names
    }


def contiguous_runs(values):
    values = list(values)
    if not values:
        return []
    runs = []
    start, current = 0, values[0]
    for index, value in enumerate(values[1:], start=1):
        if pd.isna(value) and pd.isna(current):
            same = True
        elif pd.isna(value) or pd.isna(current):
            same = False
        else:
            same = bool(value == current)
        if not same:
            runs.append((start, index, current))
            start, current = index, value
    runs.append((start, len(values), current))
    return runs


def threew_truth(frame, relative_path, descriptions):
    entity_id = Path(relative_path).name.split("_", 1)[0]
    instance_id = Path(relative_path).stem
    event_code = int(Path(relative_path).parent.name)
    timestamps = frame["event_ts"].reset_index(drop=True)

    condition_rows = []
    for start, end, value in contiguous_runs(frame["state"]):
        if pd.isna(value):
            continue
        end_ts = (
            timestamps.iloc[end]
            if end < len(timestamps)
            else timestamps.iloc[-1] + pd.Timedelta(seconds=1)
        )
        condition_rows.append({
            "entity_id": entity_id,
            "start_ts": timestamps.iloc[start],
            "end_ts": end_ts,
            "condition_code": str(int(value)),
        })

    event_rows, interval_rows = [], []
    if event_code:
        labels = pd.to_numeric(frame["class"], errors="coerce")
        active = labels.isin([event_code, 100 + event_code])
        event_number = 0
        for start, end, is_active in contiguous_runs(active):
            if not is_active:
                continue
            segment = labels.iloc[start:end]
            steady = segment.index[segment.eq(event_code)]
            impact_ts = (
                timestamps.iloc[int(steady[0])]
                if len(steady) else pd.NaT
            )
            end_ts = (
                timestamps.iloc[end] if end < len(timestamps) else pd.NaT
            )
            fault_id = (
                f"3W-{instance_id}-{event_code}-{event_number:02d}"
            )
            event_rows.append({
                "fault_id": fault_id,
                "fault_type": descriptions[event_code],
                "domain_id": entity_id,
                "onset_ts": timestamps.iloc[start],
                "observable_ts": timestamps.iloc[start],
                "impact_ts": impact_ts,
                "end_ts": end_ts,
                "group_id": pd.NA,
            })
            interval_rows.append({
                "fault_id": fault_id,
                "entity_id": entity_id,
                "start_ts": timestamps.iloc[start],
                "end_ts": (
                    end_ts if pd.notna(end_ts)
                    else timestamps.iloc[-1] + pd.Timedelta(seconds=1)
                ),
            })
            event_number += 1

    return event_rows, interval_rows, condition_rows


In [ ]:
def build_threew_pack(
    source,
    destination,
    selected_files,
    *,
    include_evaluation=True,
    batch_rows=5_000,
):
    source, destination = Path(source), Path(destination)
    descriptions = event_descriptions(source)
    rename_metrics = dict(zip(
        metric_map["native_field"], metric_map["metric_id"]
    ))

    with immutable_directory(destination) as pack:
        core = pack / "PACK-CORE"
        observations = core / "observations"
        observations.mkdir(parents=True)

        registry_rows = []
        event_rows, interval_rows, condition_rows = [], [], []
        part_number = 0

        for relative_path in selected_files:
            path = source / relative_path
            native = pd.read_parquet(path).reset_index()
            if "timestamp" not in native:
                raise ValueError(f"No timestamp index in {relative_path}")
            native = native.rename(columns={"timestamp": "event_ts"})
            native["event_ts"] = pd.to_datetime(
                native["event_ts"], utc=True
            )
            entity_id = path.name.split("_", 1)[0]

            for start in range(0, len(native), batch_rows):
                batch = native.iloc[start:start + batch_rows]
                wide = batch[
                    ["event_ts", *metric_map["native_field"]]
                ].rename(columns=rename_metrics)
                wide.insert(1, "entity_id", entity_id)
                wide = wide[
                    ["event_ts", "entity_id", *metric_map["metric_id"]]
                ]
                wide.to_parquet(
                    observations / f"part-{part_number:05d}.parquet",
                    index=False,
                    compression="zstd",
                )
                part_number += 1

            registry_rows.append({
                "entity_id": entity_id,
                "entity_type": "oil_well",
                "valid_from": native["event_ts"].min(),
                "valid_to": (
                    native["event_ts"].max()
                    + pd.Timedelta(seconds=1)
                ),
            })
            if include_evaluation:
                events, intervals, conditions = threew_truth(
                    native, relative_path, descriptions
                )
                event_rows.extend(events)
                interval_rows.extend(intervals)
                condition_rows.extend(conditions)

        metric_map[PACK_METRIC_SCHEMA].to_parquet(
            core / "metric_catalogue.parquet", index=False
        )
        pd.DataFrame(registry_rows)[
            ["entity_id", "entity_type", "valid_from", "valid_to"]
        ].to_parquet(core / "entity_registry.parquet", index=False)
        pd.DataFrame(columns=[
            "parent_entity_id",
            "child_entity_id",
            "relation_type",
        ]).to_parquet(core / "entity_relations.parquet", index=False)

        evaluation_tables = []
        if include_evaluation:
            evaluation = pack / "PACK-EVAL"
            evaluation.mkdir()
            tables = {
                "fault_events": pd.DataFrame(
                    event_rows, columns=EVAL_SCHEMAS["fault_events"]
                ),
                "fault_entity_intervals": pd.DataFrame(
                    interval_rows,
                    columns=EVAL_SCHEMAS["fault_entity_intervals"],
                ),
                "condition_states": pd.DataFrame(
                    condition_rows,
                    columns=EVAL_SCHEMAS["condition_states"],
                ),
            }
            evaluation_tables = list(tables)
            for name, frame in tables.items():
                frame.to_parquet(
                    evaluation / f"{name}.parquet", index=False
                )

        source_files = [
            source_record(source / "dataset.ini", source),
            *[
                source_record(source / relative, source)
                for relative in selected_files
            ],
        ]
        finalise_pack(
            pack,
            sector="petrobras_3w",
            pack_version="0.1.0",
            expected_cadence_seconds=1,
            source_manifest={
                "source_id": "petrobras-3w-2.0.0",
                "source_root": str(source),
                "files": source_files,
                "selected_files": list(selected_files),
                "selection_rule": (
                    "hash-pinned real-well contract challenge fixture"
                ),
                "fields_routed_only_to_evaluation": [
                    "class", "state"
                ],
            },
            evaluation_tables=evaluation_tables,
            notes=[
                "No topology, tickets, severity, or cause groups were invented.",
                "The source class is event-shaped; state is interval-shaped.",
            ],
        )

    return read_json(destination / "pack_manifest.json")


if RUN_BUILD:
    pack_manifest = build_threew_pack(
        SOURCE,
        PACK_ROOT,
        SELECTED_FILES,
        include_evaluation=True,
        batch_rows=BATCH_ROWS,
    )
else:
    pack_manifest = read_json(PACK_ROOT / "pack_manifest.json")

display(pd.Series(
    pack_manifest["core_row_counts"], name="rows"
).to_frame())
display(pd.Series(
    pack_manifest["evaluation_row_counts"], name="rows"
).to_frame())


## 5. Prove that embedded 3W labels cannot change `PACK-CORE`

This test is especially important for 3W because observations and labels
arrive in the same Parquet files.

The redacted fixture physically removes both `class` and `state`, runs the pack
translation again, and compares logical `PACK-CORE` hashes. A deliberately
leaky signature based on `class` must fail on the redacted source.


In [ ]:
def make_threew_redacted_source(source, destination, selected_files):
    destination.mkdir()
    shutil.copy2(source / "dataset.ini", destination / "dataset.ini")
    for relative_path in selected_files:
        target = destination / relative_path
        target.parent.mkdir(parents=True, exist_ok=True)
        frame = pd.read_parquet(source / relative_path)
        frame = frame.drop(columns=["class", "state"])
        frame.to_parquet(target)


def deliberately_leaky_threew_signature(source, selected_files):
    values = []
    for relative_path in selected_files:
        path = source / relative_path
        if "class" not in pq.ParquetFile(path).schema_arrow.names:
            return "missing"
        values.append(pd.read_parquet(path, columns=["class"]))
    return str(pd.concat(values)["class"].value_counts().to_dict())


with tempfile.TemporaryDirectory() as temporary:
    temporary = Path(temporary)
    redacted_source = temporary / "native_redacted"
    make_threew_redacted_source(
        SOURCE, redacted_source, SELECTED_FILES
    )

    original_pack = temporary / "pack_original"
    redacted_pack = temporary / "pack_redacted"
    build_threew_pack(
        SOURCE,
        original_pack,
        SELECTED_FILES,
        include_evaluation=True,
        batch_rows=BATCH_ROWS,
    )
    build_threew_pack(
        redacted_source,
        redacted_pack,
        SELECTED_FILES,
        include_evaluation=False,
        batch_rows=BATCH_ROWS,
    )

    original_hashes = pack_core_hashes(original_pack)
    redacted_hashes = pack_core_hashes(redacted_pack)
    assert original_hashes == redacted_hashes
    assert (
        deliberately_leaky_threew_signature(SOURCE, SELECTED_FILES)
        != deliberately_leaky_threew_signature(
            redacted_source, SELECTED_FILES
        )
    )

print("PASS — PACK-CORE is invariant after class and state are removed")
print("PASS — the negative control detects the removed labels")


## 6. Contract-fit result and handoff

The same pack interface represents telecom and 3W observations without a
sector branch. That is evidence for the abstraction, but not a claim that 3W
contains facts it does not provide.

Notebook 01B can now consume this pack unchanged. Set `ADAPTER_PACK_ROOT` to
the printed path.


In [ ]:
contract_fit = {
    "represented_cleanly": [
        "one-second multivariate telemetry",
        "well identity and observed validity",
        "source nulls retained for quality coding",
        "event labels physically isolated",
        "condition states represented as intervals",
    ],
    "not_expressible_without_invention": [
        "shared manifold topology",
        "cross-well cause groups",
        "operator tickets or maintenance actions",
        "business-impact severity",
    ],
    "deliberately_not_added": [
        "invented relations",
        "severity ordinal",
        "terminal outcomes",
    ],
}
display(pd.Series(contract_fit, name="finding").to_frame())
print("Pack root:", PACK_ROOT)
print("Next: 01B_COMMON_CANONICAL_ADAPTER.ipynb")
